# Building Image Generation Applications (Azure OpenAI)

There's more to LLMs than text generation. You can also generate images from text descriptions. Images as a modality are useful across MedTech, architecture, tourism, game development, marketing, and more. In this lesson we work with today's **GPT Image** models on Microsoft Foundry.

## Learning goals

By the end of this lesson you'll be able to:

- Explain what image generation is and where it's useful.
- Understand the `gpt-image` model family and how it differs from the legacy DALL·E models.
- Build an image generation application and edit images.

## What is image generation?

Image generation models create images from a text prompt. Modern models such as `gpt-image` learn the relationship between text and images during training, then iteratively turn random noise into an image that matches your prompt.

Two well-known families of image models are:

- **`gpt-image` (OpenAI)** — the current generation used in this lesson. It supports text-to-image generation and image editing (inpainting with a mask).
- **Midjourney** — a popular third-party model with its own service and Discord-based workflow.

> The older OpenAI image models — **DALL·E 2** and **DALL·E 3** — are legacy. DALL·E 3 is no longer available for new deployments, and features like `create_variation` only existed in DALL·E 2. Use the `gpt-image` models for new applications.

On Microsoft Foundry, **`gpt-image-2`** is the latest and most capable image model and is the recommended default. `gpt-image-1.5` and `gpt-image-1-mini` are also generally available.

> **Important:** `gpt-image` models return the generated image as **base64** (`b64_json`), not as a URL. Your code decodes the base64 string to bytes and saves it — there's no image URL to download.


## Costruire la tua prima applicazione di generazione immagini

Cosa serve quindi per costruire un'applicazione di generazione immagini? Hai bisogno delle seguenti librerie:

- **python-dotenv**, si consiglia vivamente di usare questa libreria per tenere i tuoi segreti in un file *.env* separato dal codice.
- **openai**, questa libreria è quella che userai per interagire con l'API di OpenAI.
- **pillow**, per lavorare con le immagini in Python.

Se non l'hai ancora fatto, segui le istruzioni nella pagina [Microsoft Learn](https://learn.microsoft.com/azure/ai-foundry/openai/how-to/create-resource?pivots=web-portal&WT.mc_id=academic-105485-koreyst) per creare una risorsa Microsoft Foundry e un modello. Seleziona **gpt-image-2** come modello (l'ultimo modello di immagini Azure OpenAI; DALL·E è legacy).

1. Crea un file *.env* con il seguente contenuto:

    ```text
    AZURE_OPENAI_ENDPOINT=<your endpoint>
    AZURE_OPENAI_API_KEY=<your key>
    AZURE_OPENAI_DEPLOYMENT="gpt-image-2"
    ```

    Trova queste informazioni nel portale Microsoft Foundry per la tua risorsa nella sezione "Deployments".



1. Raccogli le librerie sopra in un file chiamato *requirements.txt* come segue:

    ```text
    python-dotenv
    openai
    pillow
    ```


1. Successivamente, crea un ambiente virtuale e installa le librerie:


In [ ]:
# create virtual env
! python3 -m venv venv
# activate environment
! source venv/bin/activate
# install libraries
# pip install -r requirements.txt, if using a requirements.txt file 
! pip install python-dotenv openai pillow

> [!NOTE]
> Per Windows, usa i seguenti comandi per creare e attivare il tuo ambiente virtuale:

    ```bash
    python3 -m venv venv
    venv\Scripts\activate.bat
    ````

1. Aggiungi il seguente codice in un file chiamato *app.py*:

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    from openai import AzureOpenAI, BadRequestError

    # import dotenv
    dotenv.load_dotenv()

    # configura il client Azure OpenAI (Microsoft Foundry).
    # I modelli di immagini necessitano di una versione API recente — verifica la documentazione di Foundry per quella richiesta dal tuo modello.
    client = AzureOpenAI(
      azure_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"],
      api_key=os.environ['AZURE_OPENAI_API_KEY'],
      api_version = "2025-04-01-preview"
      )

    try:
        # Crea un'immagine utilizzando l'API di generazione immagini
        generation_response = client.images.generate(
            prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Inserisci qui il testo del tuo prompt
            size='1024x1024',
            n=1,
            model=os.environ['AZURE_OPENAI_DEPLOYMENT']   # es. "gpt-image-2"
        )
        # Imposta la directory per l'immagine salvata
        image_dir = os.path.join(os.curdir, 'images')

        # Se la directory non esiste, creala
        if not os.path.isdir(image_dir):
            os.mkdir(image_dir)

        # Inizializza il percorso dell'immagine (nota che il tipo di file dovrebbe essere png)
        image_path = os.path.join(image_dir, 'generated-image.png')

        # I modelli gpt-image restituiscono l'immagine come base64 (b64_json), non come URL
        image_b64 = generation_response.data[0].b64_json
        generated_image = base64.b64decode(image_b64)
        with open(image_path, "wb") as image_file:
            image_file.write(generated_image)

        # Visualizza l'immagine nel visualizzatore immagini predefinito
        image = Image.open(image_path)
        image.show()

    # gestisci le eccezioni
    except BadRequestError as err:
        print(err)

    ```

Spieghiamo questo codice:

- Per prima cosa, importiamo le librerie di cui abbiamo bisogno, tra cui la libreria OpenAI, la libreria dotenv, il modulo base64 e la libreria Pillow.

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    from openai import AzureOpenAI, BadRequestError
    ```

- Poi, carichiamo le variabili d'ambiente dal file *.env*.

    ```python
    # importa dotenv
    dotenv.load_dotenv()
    ```

- Successivamente, configuriamo il client Azure OpenAI (Microsoft Foundry).

    ```python
    client = AzureOpenAI(
      azure_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"],
      api_key=os.environ['AZURE_OPENAI_API_KEY'],
      api_version = "2025-04-01-preview"
      )
    ```

- Poi, generiamo l'immagine:

    ```python
    generation_response = client.images.generate(
        prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Inserisci qui il testo del tuo prompt
        size='1024x1024',
        n=1,
        model=os.environ['AZURE_OPENAI_DEPLOYMENT']
    )
    ```

    I modelli `gpt-image` restituiscono l'immagine come stringa **base64** in `data[0].b64_json`. La decodifichiamo in byte e la scriviamo su un file — non esiste un URL per il download.

    ```python
    image_b64 = generation_response.data[0].b64_json
    generated_image = base64.b64decode(image_b64)
    with open(image_path, "wb") as image_file:
        image_file.write(generated_image)
    ```

- Infine, apriamo l'immagine e usiamo il visualizzatore immagini standard per mostrarla:

    ```python
    image = Image.open(image_path)
    image.show()
    ```

### Ulteriori dettagli sulla generazione dell'immagine

Diamo un'occhiata ai parametri di `images.generate`:

- **prompt**, è il testo utilizzato per generare l'immagine. Qui è "Coniglio su cavallo, che tiene un lecca-lecca, in un prato nebbioso dove crescono giunchiglie".
- **size**, è la dimensione dell'immagine generata (per esempio `1024x1024`, `1536x1024`, `1024x1536` o `"auto"`).
- **n**, è il numero di immagini generate. Qui ne generiamo una.
- **model**, è il nome del deployment del modello di immagine (per esempio `gpt-image-2`).

> I modelli di immagine non prendono un parametro `temperature` — quello è un controllo per la generazione di testi. Per ottenere varietà, chiama di nuovo l'API; per ridurre la varietà, rendi il tuo prompt più specifico.

## Capacità aggiuntive della generazione di immagini

Hai visto come generare un'immagine con poche righe di Python. I modelli `gpt-image` possono anche **modificare** un'immagine esistente. Fornendo un'immagine, una **mask** opzionale (che evidenzia l'area da modificare) e un prompt, puoi alterare una parte di un'immagine — per esempio, aggiungere un cappello al nostro coniglio.

```python
response = client.images.edit(
  model=os.environ['AZURE_OPENAI_DEPLOYMENT'],
  image=open("base_image.png", "rb"),
  mask=open("mask.png", "rb"),
  prompt="An image of a rabbit with a hat on its head.",
  n=1,
  size="1024x1024"
)
# le modifiche vengono restituite anche in base64
edited_image = base64.b64decode(response.data[0].b64_json)
```

L'immagine base contiene solo il coniglio; l'immagine finale ha il cappello sul coniglio.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:
Questo documento è stato tradotto utilizzando il servizio di traduzione AI [Co-op Translator](https://github.com/Azure/co-op-translator). Sebbene ci impegniamo per garantire la precisione, si prega di notare che le traduzioni automatizzate possono contenere errori o imprecisioni. Il documento originale nella sua lingua nativa deve essere considerato la fonte autorevole. Per informazioni critiche, si raccomanda una traduzione professionale effettuata da un essere umano. Non siamo responsabili per eventuali malintesi o interpretazioni errate derivanti dall’uso di questa traduzione.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
